# PoshanScan upper-arm segmentation on Colab

This notebook runs the split-preserving training and held-out evaluation workflow in `scripts/train_arm_segmentation_colab.py`. It uses pretrained YOLOv8n-seg transfer learning, saves best/last checkpoints and reports, then packages the artifacts for download. No result is prefilled or fabricated.

Before running, choose **Runtime → Change runtime type → GPU**. Colab GPU availability and type vary. The script checks CUDA and falls back to CPU with batch size 1 if needed.

Upload these two files to the same Google Drive folder: `poshanscan_colab_dataset.zip` and `train_arm_segmentation_colab.py`. The dataset archive is about 341 MB. It contains the 120 active unique split images, their 120 labels, the unchanged split files, manifest and reports. The 11 exact duplicate aliases remain preserved in your local project but are excluded from the splits and bundle.

In [ ]:
import sys, torch, torchvision
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('CPU fallback will be used; training will take longer.')

## Locate the workflow and dataset

The training script installs Ultralytics and any missing non-PyTorch requirements using `--no-deps`. It verifies the installed torch/torchvision pair and will stop rather than replace either package. If Colab's existing pair is broken, reconnect to a fresh GPU runtime instead of pip-installing PyTorch.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess

drive.mount('/content/drive')
DRIVE_FOLDER = Path('/content/drive/MyDrive/PoshanScanColab')
DATASET_ZIP = DRIVE_FOLDER / 'poshanscan_colab_dataset.zip'
WORKFLOW_SCRIPT = DRIVE_FOLDER / 'train_arm_segmentation_colab.py'
if not DATASET_ZIP.is_file():
    raise FileNotFoundError(DATASET_ZIP)
if not WORKFLOW_SCRIPT.is_file():
    raise FileNotFoundError(WORKFLOW_SCRIPT)
print('Dataset:', DATASET_ZIP)
print('Workflow:', WORKFLOW_SCRIPT)

## Train and evaluate

This reads and verifies the existing 84/24/12 split, then copies its images/labels into a Colab-local YOLO layout. It does not regenerate split lists. Training uses train and validation only; test metrics are computed afterward on the reserved 12 images.

In [ ]:
subprocess.run([
    sys.executable, str(WORKFLOW_SCRIPT),
    '--dataset-zip', str(DATASET_ZIP),
    '--output-dir', '/content/poshanscan_colab_output',
    '--weights', 'yolov8n-seg.pt',
    '--epochs', '100', '--imgsz', '416', '--seed', '2026',
], check=True)

## Download the completed artifacts

The full ZIP contains best/last checkpoints, Ultralytics training logs and plots, training configuration, validation/test reports, per-image metrics, compatibility metadata and comparison images. You can also download just `best.pt`. These files appear only after the training/evaluation cell completes.

In [ ]:
from google.colab import files
files.download('/content/poshanscan_colab_artifacts.zip')

In [ ]:
from google.colab import files
files.download('/content/poshanscan_colab_output/best.pt')

## Return best.pt to PoshanScan

Copy the downloaded checkpoint to `poshanscan/ml-service/app/models/arm_segmentation/best.pt`. The workflow verifies that the checkpoint task is `segment` and its only class is `upper_arm`. For reliable loading, use the Ultralytics version recorded in `checkpoint_compatibility.json` in the ML service deployment. The current FastAPI integration then loads the checkpoint at startup; the API contract and MUAC geometric calculation remain unchanged. Segmentation scores on synthetic images are not clinical validation.